# Image Processing Algorithms Comparison

Companion notebook to the BI New Vision internship report (Section 3.4).

Runs entirely on the free Google Colab CPU tier. Each section compares several classical algorithms
for one step of the OCR preprocessing pipeline, printing timing / PSNR / SSIM and showing a visual grid.

Upload any test image (a scanned or photographed document works best) when prompted in Cell 0.

## Cell 0 — Setup and image loading

In [ ]:
!pip install -q opencv-python-headless scikit-image scipy PyWavelets numpy matplotlib

import cv2
import numpy as np
import matplotlib.pyplot as plt
import time
from skimage.filters import threshold_otsu, threshold_sauvola, threshold_niblack
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
from skimage.restoration import denoise_wavelet

try:
    from google.colab import files
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    img_color = cv2.imread(fname)
except ImportError:
    import urllib.request
    url = "https://raw.githubusercontent.com/opencv/opencv/master/samples/data/text_defocus.jpg"
    urllib.request.urlretrieve(url, "sample.jpg")
    img_color = cv2.imread("sample.jpg")

img = cv2.cvtColor(img_color, cv2.COLOR_BGR2GRAY)
plt.figure(figsize=(6, 6))
plt.imshow(img, cmap="gray")
plt.title("Original")
plt.axis("off")
plt.show()

## Cell 1 — Denoising comparison (Gaussian, Median, Bilateral, Non-Local Means, Wavelet)

In [ ]:
def add_gaussian_noise(image, sigma=15):
    noisy = image.astype(np.float32) + np.random.normal(0, sigma, image.shape)
    return np.clip(noisy, 0, 255).astype(np.uint8)

clean = img.copy()
noisy = add_gaussian_noise(clean, sigma=15)

def gaussian_denoise(im):  return cv2.GaussianBlur(im, (5, 5), 1.5)
def median_denoise(im):    return cv2.medianBlur(im, 5)
def bilateral_denoise(im): return cv2.bilateralFilter(im, 9, 75, 75)
def nlm_denoise(im):       return cv2.fastNlMeansDenoising(im, None, h=10,
                                   templateWindowSize=7, searchWindowSize=21)
def wavelet_denoise(im):
    out = denoise_wavelet(im, rescale_sigma=True, mode="soft")
    return (out * 255).astype(np.uint8) if out.max() <= 1.0 else out.astype(np.uint8)

denoise_methods = {
    "Noisy input":      lambda im: im,
    "Gaussian":          gaussian_denoise,
    "Median":            median_denoise,
    "Bilateral":         bilateral_denoise,
    "Non-Local Means":   nlm_denoise,
    "Wavelet (soft)":    wavelet_denoise,
}

results = {}
for name, fn in denoise_methods.items():
    t0 = time.time()
    out = fn(noisy)
    dt = time.time() - t0
    p, s = psnr(clean, out), ssim(clean, out)
    results[name] = (out, dt, p, s)
    print(f"{name:18s} | time: {dt*1000:6.1f} ms | PSNR: {p:5.2f} dB | SSIM: {s:5.3f}")

fig, axes = plt.subplots(1, len(results), figsize=(4 * len(results), 4))
for ax, (name, (out, dt, p, s)) in zip(axes, results.items()):
    ax.imshow(out, cmap="gray")
    ax.set_title(f"{name}\nPSNR={p:.1f} SSIM={s:.2f}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## Cell 2 — Sharpening comparison (Unsharp mask, Laplacian, High-boost)

In [ ]:
def unsharp_mask(im, k=1.5, sigma=2):
    blurred = cv2.GaussianBlur(im, (0, 0), sigma)
    return cv2.addWeighted(im, 1 + k, blurred, -k, 0)

def laplacian_sharpen(im, c=1.0):
    lap = cv2.Laplacian(im, cv2.CV_64F)
    out = im.astype(np.float64) - c * lap
    return np.clip(out, 0, 255).astype(np.uint8)

def high_boost(im, A=1.5, sigma=2):
    blurred = cv2.GaussianBlur(im, (0, 0), sigma)
    mask = im.astype(np.float64) - blurred.astype(np.float64)
    out = A * im.astype(np.float64) + mask
    return np.clip(out, 0, 255).astype(np.uint8)

sharp_methods = {
    "Original":            lambda im: im,
    "Unsharp Mask":        unsharp_mask,
    "Laplacian sharpen":   laplacian_sharpen,
    "High-boost (A=1.5)":  high_boost,
}

fig, axes = plt.subplots(1, len(sharp_methods), figsize=(4 * len(sharp_methods), 4))
for ax, (name, fn) in zip(axes, sharp_methods.items()):
    ax.imshow(fn(img), cmap="gray")
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Cell 3 — Binarisation comparison (Otsu, Sauvola, Niblack, adaptive Gaussian)

In [ ]:
def binarize_otsu(im):
    t = threshold_otsu(im)
    return (im > t).astype(np.uint8) * 255

def binarize_sauvola(im, window=25, k=0.2):
    t = threshold_sauvola(im, window_size=window, k=k)
    return (im > t).astype(np.uint8) * 255

def binarize_niblack(im, window=25, k=-0.2):
    t = threshold_niblack(im, window_size=window, k=k)
    return (im > t).astype(np.uint8) * 255

def binarize_adaptive_gaussian(im, block_size=25, C=10):
    return cv2.adaptiveThreshold(im, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                  cv2.THRESH_BINARY, block_size, C)

bin_methods = {
    "Otsu (global)":     binarize_otsu,
    "Sauvola (local)":   binarize_sauvola,
    "Niblack (local)":   binarize_niblack,
    "Adaptive Gaussian": binarize_adaptive_gaussian,
}

fig, axes = plt.subplots(1, len(bin_methods), figsize=(4 * len(bin_methods), 4))
for ax, (name, fn) in zip(axes, bin_methods.items()):
    t0 = time.time()
    out = fn(img)
    dt = time.time() - t0
    ax.imshow(out, cmap="gray")
    ax.set_title(f"{name}\n{dt*1000:.1f} ms")
    ax.axis("off")
plt.tight_layout()
plt.show()

## Cell 4 — Interpolation comparison (Nearest, Bilinear, Bicubic, Lanczos)

In [ ]:
h, w = img.shape
small = cv2.resize(img, (w // 2, h // 2), interpolation=cv2.INTER_AREA)

interp_methods = {
    "Nearest":  cv2.INTER_NEAREST,
    "Bilinear": cv2.INTER_LINEAR,
    "Bicubic":  cv2.INTER_CUBIC,
    "Lanczos4": cv2.INTER_LANCZOS4,
}

fig, axes = plt.subplots(1, len(interp_methods), figsize=(4 * len(interp_methods), 4))
for ax, (name, flag) in zip(axes, interp_methods.items()):
    up = cv2.resize(small, (w, h), interpolation=flag)
    p, s = psnr(img, up), ssim(img, up)
    ax.imshow(up, cmap="gray")
    ax.set_title(f"{name}\nPSNR={p:.1f} SSIM={s:.2f}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## Cell 5 (optional) — Learned super-resolution vs. bicubic

Downloads small pretrained CPU-friendly super-resolution models (ESPCN, FSRCNN) and compares them to classical bicubic upscaling.
Skip this cell if the download URLs are unreachable in your environment.

In [ ]:
!wget -q -O ESPCN_x2.pb https://github.com/fannymonori/TF-ESPCN/raw/master/export/ESPCN_x2.pb
!wget -q -O FSRCNN_x2.pb https://github.com/Saafke/FSRCNN_Tensorflow/raw/master/models/FSRCNN_x2.pb

sr = cv2.dnn_superres.DnnSuperResImpl_create()
small_color = cv2.cvtColor(small, cv2.COLOR_GRAY2BGR)

for name, path, alg, scale in [("ESPCN", "ESPCN_x2.pb", "espcn", 2),
                                ("FSRCNN", "FSRCNN_x2.pb", "fsrcnn", 2)]:
    sr.readModel(path)
    sr.setModel(alg, scale)
    up_color = sr.upsample(small_color)
    up = cv2.cvtColor(up_color, cv2.COLOR_BGR2GRAY)
    p, s = psnr(img, up), ssim(img, up)
    print(f"{name}: PSNR={p:.2f} dB, SSIM={s:.3f}")

## Cell 6 — Deskew comparison (Hough vs. projection profile)

In [ ]:
def rotate_image(im, angle):
    h, w = im.shape
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    return cv2.warpAffine(im, M, (w, h), borderValue=255)

def deskew_hough(im):
    edges = cv2.Canny(im, 50, 150)
    lines = cv2.HoughLines(edges, 1, np.pi / 180, 150)
    if lines is None:
        return 0.0
    angles = [(theta * 180 / np.pi) - 90 for rho, theta in lines[:, 0]]
    return float(np.median(angles))

def deskew_projection(im, angle_range=10, step=0.2):
    best_angle, best_score = 0.0, -1.0
    for angle in np.arange(-angle_range, angle_range, step):
        rotated = rotate_image(im, angle)
        proj = np.sum(rotated < 128, axis=1)
        score = np.var(proj)
        if score > best_score:
            best_score, best_angle = score, angle
    return best_angle

true_angle = 4.0
test_img = rotate_image(img, -true_angle)

angle_hough = deskew_hough(test_img)
angle_proj = deskew_projection(test_img)

print(f"True angle:                  {true_angle:.2f} deg")
print(f"Hough estimate:               {angle_hough:.2f} deg (error {abs(angle_hough - true_angle):.2f})")
print(f"Projection profile estimate:  {angle_proj:.2f} deg (error {abs(angle_proj - true_angle):.2f})")